# Chapter 11 - HMM Demo

This demo shows how to define parameters of a Hidden Markov Model and how to run the Viterbi algorithm on an example 4-day observation sequence. The structure follows the style of the exercise solution file.

In [ ]:
# Định nghĩa tham số mô hình HMM

# Tập trạng thái ẩn
states = ["angry", "happy"]

# Tập quan sát có thể xuất hiện
observations = ["silent", "talking", "shouting"]

# Xác suất khởi tạo (ngày 1 bắt đầu ở mỗi trạng thái)
pi = {
    "angry": 0.4,
    "happy": 0.6
}

# Ma trận chuyển trạng thái A[i][j]: xác suất từ trạng thái i sang j
A = {
    "angry": {"angry": 0.7, "happy": 0.3},
    "happy": {"angry": 0.4, "happy": 0.6}
}

# Ma trận phát B[j][o]: xác suất trạng thái j sinh ra quan sát o
B = {
    "angry":  {"silent": 0.3, "talking": 0.5, "shouting": 0.2},
    "happy":  {"silent": 0.1, "talking": 0.6, "shouting": 0.3}
}

# Chuỗi quan sát trong 4 ngày
O = ["silent", "talking", "silent", "shouting"]

In [ ]:
# Thuật toán Viterbi

def viterbi(O, states, pi, A, B):
    # V[t][s] lưu xác suất cao nhất để đi đến trạng thái s tại thời điểm t
    V = [{}]

    # path[s] lưu chuỗi trạng thái tốt nhất dẫn đến trạng thái s
    path = {}

    # Bước khởi tạo (t = 0)
    for s in states:
        # V_0(s) = pi[s] * B[s][O[0]]
        V[0][s] = pi[s] * B[s][O[0]]
        path[s] = [s]

    # Bước đệ quy: tính từ t = 1 đến t = len(O)-1
    for t in range(1, len(O)):
        V.append({})
        new_path = {}

        for j in states:
            # Tìm trạng thái i* ở t-1 cho xác suất: V[t-1][i] * A[i][j] lớn nhất
            (prob, best_state) = max((V[t-1][i] * A[i][j], i) for i in states)

            # Nhân thêm xác suất phát B[j][O[t]]
            V[t][j] = prob * B[j][O[t]]

            # Cập nhật đường đi tốt nhất dẫn tới j
            new_path[j] = path[best_state] + [j]

        path = new_path

    # Bước kết thúc: chọn trạng thái kết thúc có xác suất cao nhất
    final_state = max(states, key=lambda s: V[-1][s])

    return V, final_state, path

In [ ]:
# Chạy thuật toán Viterbi cho chuỗi quan sát O
V, final_state, paths = viterbi(O, states, pi, A, B)

print("Trạng thái cuối cùng có xác suất cao nhất:", final_state)
print("Chuỗi trạng thái ẩn hợp lý nhất:", paths[final_state])

In [ ]:
# Hiển thị bảng xác suất Viterbi theo từng ngày
import pandas as pd

df = pd.DataFrame(V)
df.index = ["Day 1", "Day 2", "Day 3", "Day 4"]

df

In [ ]:
# ==========================
# Dự đoán trạng thái ngày 5 và ngày 6
# ==========================

import numpy as np

# Chuyển ma trận A sang dạng numpy
A_mat = np.array([
    [A["angry"]["angry"], A["angry"]["happy"]],
    [A["happy"]["angry"], A["happy"]["happy"]]
])

# Lấy phân phối trạng thái ở ngày 4 từ bảng Viterbi
# Ta chuẩn hóa xác suất ở ngày cuối để thành phân phối hợp lệ
last_day = V[-1]
posterior_day4 = np.array([last_day["angry"], last_day["happy"]], dtype=float)
posterior_day4 = posterior_day4 / posterior_day4.sum()

print("Phân phối trạng thái ngày 4 (angry, happy):", posterior_day4)

# Dự đoán ngày 5
posterior_day5 = posterior_day4 @ A_mat
print("Phân phối trạng thái ngày 5 (angry, happy):", posterior_day5)

# Dự đoán ngày 6
posterior_day6 = posterior_day5 @ A_mat
print("Phân phối trạng thái ngày 6 (angry, happy):", posterior_day6)
